<span style="font-size: 36px; color: White; font-family: 'Times New Roman';">**Secretaria de Desenvolvimento Econômico de Pernambuco – SEAIEE**</span>

<span style="font-size: 18px; color: White; font-family: 'Times New Roman';">Caio G. V. Coutinho</span>

---

# PRODEPE e PROIND – **mapa interativo por município**

Lê a planilha de novos investimentos da ADEPE e gera **um único arquivo HTML**
(`docs/index.html`), pronto para o GitHub Pages, com o mapa de calor de Pernambuco e filtros
dinâmicos. Nada depende de servidor: os dados agregados e a malha municipal vão dentro da própria página.

| Etapa | O que faz |
|---|---|
| 1. Leitura e limpeza | padroniza município, região de desenvolvimento (RD), segmento e natureza do projeto |
| 2. Anonimização | o nome da empresa é usado apenas para contar empresas distintas e **é descartado**; a página recebe só um código aleatório |
| 3. Deflação | valores em R$ levados a preços do último ano da planilha de deflatores (IPCA) |
| 4. Malha | shapefile municipal de PE, simplificado e convertido em caminhos SVG |
| 5. Site | mapa + filtros em lista suspensa, responsivos (indicador, programa, ano, segmento, tipo, natureza, RD, empreendimento novo) |
| 6. Verificação | confere que nenhum nome de empresa aparece no HTML gerado |

O layout segue o padrão de `ovinocaprino_04_mapas_post.ipynb` (**fundo escuro** por padrão, com opção de **tema claro**; a página respeita a preferência do sistema e lembra a escolha): título em branco e
subtítulo em cinza, **escala contínua** resumida numa barra fina com só os valores das pontas, divisas na
cor do fundo, sem siglas sobre o mapa e fonte/data-base impressas na página.

In [ ]:
# %% Célula 1 – Configuração
import os, re, json, unicodedata, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ caminhos
# o estudo circula entre duas máquinas, com nomes de usuário diferentes.
# PRODEPE_BASE (variável de ambiente) permite apontar para outra pasta, se preciso.
CANDIDATOS_BASE = [
    os.environ.get("PRODEPE_BASE", ""),
    r"C:\Users\CaioC\OneDrive\Work\SDEC\Estudos Econômicos\data",
    r"C:\Users\Caio.coutinho\OneDrive\Work\SDEC\Estudos Econômicos\data",
]
BASE = next((p for p in CANDIDATOS_BASE if p and os.path.isdir(p)), CANDIDATOS_BASE[-1])

PASTA     = os.path.join(BASE, "Marcus", "PRODEPE e Proind")
ARQ_ADEPE = os.path.join(PASTA, "PLANILHA DE NOVOS INVESTIMENTOS ADEPE.xlsx")
ABA_ADEPE = "PRODEPE + PROIND"
ARQ_DEFL  = os.path.join(BASE, "Deflatores.xlsx")
SHP_MUN   = os.path.join(BASE, "Saúde", "input", "BR_Municipios_2025")

# saída: pasta docs/ do repositório (GitHub Pages publica a partir dela)
OUT = os.environ.get("PRODEPE_OUT", os.path.join(PASTA, "prodepe_proind", "docs"))
os.makedirs(OUT, exist_ok=True)
LOGO_DIR = os.path.join(OUT, "assets")   # logo da SDEC em duas versões (fundo claro e fundo escuro)

# ------------------------------------------------------------------ identidade visual
# (mesma paleta do notebook de ovinocaprinocultura)
COR_FUNDO   = "#141414"
COR_TEXTO   = "#ffffff"
COR_SUAVE   = "#b5b5b5"
COR_AUSENTE = "#3a3a3a"
GAMA        = 0.5     # < 1 abre o contraste na parte baixa da distribuição; a barra da legenda
                      # usa a mesma transformação, então a escala continua honesta

def truncar(cmap, inicio=0.06, fim=0.92, n=48):
    """Corta as pontas da escala. A ponta clara não pode virar o elemento mais chamativo do mapa
    e a ponta escura não pode sumir no fundo #141414."""
    base = plt.get_cmap(cmap)
    return [to_hex(c) for c in base(np.linspace(inicio, fim, n))]

# tema escuro: começa quase branco e termina antes do tom que some no fundo;
# tema claro: o começo precisa ser visível sobre fundo claro (e distinto do cinza de "sem projeto")
PALETAS = [
    {"nome": nome, "cores": truncar(cm), "claro": truncar(cm, 0.25, 1.0)}
    for nome, cm in (("Azul", "Blues"), ("Vermelho", "Reds"), ("Laranja", "Oranges"))
]

SEED_ANONIMO = 20260921   # só embaralha os códigos das empresas; não permite recuperar nomes
print("base :", BASE)
print("saída:", OUT)

In [ ]:
# %% Célula 2 – Utilidades
def norm(s):
    """MAIÚSCULAS, sem acento, só alfanumérico e espaço."""
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii").upper()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

_MIN = {"DE", "DO", "DA", "DOS", "DAS", "E"}
def titulo(s):
    """Título com preposições em minúscula: 'SERTÃO DO PAJEÚ' -> 'Sertão do Pajeú'."""
    p = str(s).strip().lower().split()
    return " ".join(w if (i and w in {x.lower() for x in _MIN}) else w.capitalize() for i, w in enumerate(p))

def br(x, casas=0):
    """Número no formato brasileiro: 1.234.567 / 12,3."""
    if pd.isna(x):
        return "–"
    return f"{x:,.{casas}f}".replace(",", "@").replace(".", ",").replace("@", ".")

---
## 1. Leitura, limpeza e anonimização

Cada linha da planilha é um **projeto/decreto**. Pontos tratados:

* **RD** com valor trocado (nome de segmento digitado na coluna de RD) e grafias diferentes da mesma RD –
  a RD é reconstituída pelo município;
* **município** com erro de digitação ("São Lorenço da Mata");
* **segmento** com 30 grafias – agrupadas em categorias; projetos do PROIND não têm segmento informado;
* **natureza do projeto** agrupada em quatro classes; a coluna "Empreendimento Novo?" vira filtro próprio;
* linhas do PRODEPE sem valor de investimento/emprego (importação e centrais de distribuição) contam como
  projeto e como empresa, mas entram com zero nas somas.

In [ ]:
# %% Célula 3 – Leitura e limpeza
raw = pd.read_excel(ARQ_ADEPE, sheet_name=ABA_ADEPE)
raw.columns = [str(c).strip() for c in raw.columns]
print(f"{len(raw)} linhas lidas")

# ---------- ano e data-base
def _data(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, (int, float)):
        return pd.Timestamp("1899-12-30") + pd.Timedelta(days=float(x))
    return pd.to_datetime(x, errors="coerce")
DATA_BASE_TS = raw["DATA DO DECRETO"].map(_data).max()
MESES = ["janeiro","fevereiro","março","abril","maio","junho","julho","agosto","setembro","outubro","novembro","dezembro"]
DATA_BASE = f"{MESES[DATA_BASE_TS.month-1]} de {DATA_BASE_TS.year}"

# ---------- município (casa com a malha pelo nome normalizado)
mun = raw["MUNICÍPIO PRETENDIDO"].map(norm)
CORRIGE_MUN = {"SAO LORENCO DA MATA": "SAO LOURENCO DA MATA"}
raw["mun_norm"] = mun.replace(CORRIGE_MUN)

# ---------- região de desenvolvimento
RD_OFICIAIS = ["AGRESTE CENTRAL", "AGRESTE MERIDIONAL", "AGRESTE SETENTRIONAL", "MATA NORTE", "MATA SUL",
               "METROPOLITANA", "SERTÃO CENTRAL", "SERTÃO DE ITAPARICA", "SERTÃO DO ARARIPE",
               "SERTÃO DO MOXOTÓ", "SERTÃO DO PAJEÚ", "SERTÃO DO SÃO FRANCISCO"]
rd = (raw["RD"].astype(str).str.strip().str.upper()
      .replace({"REGIÃO METROPOLITANA": "METROPOLITANA", "SERTÃO DO FRANCISCO": "SERTÃO DO SÃO FRANCISCO"}))
rd = rd.where(rd.isin(RD_OFICIAIS))                      # o que sobrar é digitação errada
moda_rd = rd.groupby(raw["mun_norm"]).agg(lambda s: s.mode().iloc[0] if s.notna().any() else np.nan)
# município cujas únicas linhas trazem a RD errada: usa a aba "APENAS NOVOS PROJETOS" (coluna RD NORMALIZADA)
RD_MANUAL = {"JOAO ALFREDO": "AGRESTE SETENTRIONAL"}
raw["rd"] = rd.fillna(raw["mun_norm"].map(moda_rd)).fillna(raw["mun_norm"].map(RD_MANUAL))
assert raw["rd"].notna().all(), raw.loc[raw["rd"].isna(), "MUNICÍPIO PRETENDIDO"].unique()
print(f"RD reconstituída pelo município em {int(rd.isna().sum())} linhas")

# ---------- segmento
CHAVES_SEG = [("IMPORTADOR", "Comércio importador atacadista"), ("COMERCIO ATACADISTA", "Comércio atacadista"),
              ("AGROIND", "Agroindústria"), ("METALMEC", "Metalmecânica"),
              ("MINERAIS", "Minerais não metálicos"), ("CIMENTO", "Minerais não metálicos"),
              ("ELETRO", "Eletroeletrônica"), ("PLASTIC", "Plástico"), ("MOVEIS", "Móveis"),
              ("BEBIDAS", "Bebidas"), ("TEXTIL", "Têxtil"), ("FARMA", "Farmacoquímica e higiene"),
              ("FARMO", "Farmacoquímica e higiene"), ("HIGIENE", "Farmacoquímica e higiene"),
              ("QUIMICOS", "Produtos químicos"), ("FERTILIZANTES", "Produtos químicos")]
def segmento(row):
    s = norm(row["SETOR DE ATIVIDADE"])
    if not s:
        return "Não informado (PROIND)"
    rotulos = {rot for chave, rot in CHAVES_SEG if chave in s}
    if len(rotulos) > 1:
        return "Mais de um segmento"      # ex.: "Plástico / Metalmecânica"
    return rotulos.pop() if rotulos else "Outros"
raw["seg"] = raw.apply(segmento, axis=1)

# ---------- tipo de projeto e natureza
raw["tipo"] = raw["TIPO DE PROJETO"].map(titulo).replace({"Proind": "PROIND"})

def natureza(s):
    s = norm(s)
    if "IMPLANT" in s or "IMPLAT" in s:
        return "Implantação"
    if "ADES" in s or "MIGRA" in s:
        return "Adesão ou migração"
    if "AMPLIA" in s or "REVITAL" in s:
        return "Ampliação ou revitalização"
    return "Isonomia ou manutenção"
raw["nat"] = raw["NATUREZA DO PROJETO"].map(natureza)
raw["novo"] = np.where(raw["Empreendimento Novo?"].astype(str).str.strip().str.upper() == "SIM",
                       "Novo", "Já existente")
raw["prog"] = raw["PRODEPE / PROIND"].astype(str).str.strip().str.upper()

print(raw["seg"].value_counts().to_string())
print(raw["nat"].value_counts().to_string())

## 2. Deflação

A planilha de deflatores traz o fator que leva cada ano a **preços do último ano da série** (que vale 1).
O fator é aplicado pelo ano do decreto/reunião (`ANO`). Investimentos são convertidos para **R$ milhões**.

In [ ]:
# %% Célula 4 – Deflação
d = pd.read_excel(ARQ_DEFL, sheet_name="Séries")
d.columns = [str(c).strip() for c in d.columns]
d = d.rename(columns={d.columns[0]: "ano"})
d["ano"] = pd.to_numeric(d["ano"], errors="coerce")
d = d.dropna(subset=["ano"]).astype({"ano": int})
DEFL = d.set_index("ano")["Deflator"].astype(float)
ANO_BASE = int(DEFL.index.max())
assert abs(DEFL[ANO_BASE] - 1) < 1e-9, "o deflator do último ano deveria ser 1"
assert set(raw["ANO"].unique()) <= set(DEFL.index), "há ano sem deflator"

raw["inv_nominal"] = pd.to_numeric(raw["INVESTIMENTOS (R$)"], errors="coerce").fillna(0.0)
raw["inv_real_mi"] = raw["inv_nominal"] * raw["ANO"].map(DEFL) / 1e6
raw["job"] = pd.to_numeric(raw["EMPREGOS"], errors="coerce").fillna(0).astype(int)

print(f"preços de {ANO_BASE}")
print(DEFL.loc[raw["ANO"].min():].round(4).to_string())
print(f"investimento total: R$ {br(raw['inv_nominal'].sum()/1e6, 1)} mi nominais -> "
      f"R$ {br(raw['inv_real_mi'].sum(), 1)} mi de {ANO_BASE}")

## 3. Malha municipal

Recorte de Pernambuco do shapefile do IBGE (2025). Fernando de Noronha sai da malha: o polígono oceânico
estica o mapa e encolhe o continente (mesmo tratamento do mapa de ovinocaprinocultura, cujo caso é idêntico).
A geometria é simplificada preservando a vizinhança entre municípios e convertida em caminhos SVG.

In [ ]:
# %% Célula 5 – Malha de Pernambuco em SVG
shp = [os.path.join(SHP_MUN, f) for f in os.listdir(SHP_MUN) if f.lower().endswith(".shp")][0]
try:
    geo = gpd.read_file(shp, engine="pyogrio", where="SIGLA_UF = 'PE'")
except Exception:
    geo = gpd.read_file(shp)
    geo = geo[geo["SIGLA_UF"] == "PE"]
geo = geo[geo["NM_MUN"].map(norm) != "FERNANDO DE NORONHA"].copy()
geo["mun_norm"] = geo["NM_MUN"].map(norm)
geo = geo.sort_values("NM_MUN").reset_index(drop=True)
print(len(geo), "municípios na malha")

# todo município da planilha precisa existir na malha
faltam = sorted(set(raw["mun_norm"]) - set(geo["mun_norm"]))
assert not faltam, f"municípios sem correspondência na malha: {faltam}"

# simplificação com preservação de topologia entre vizinhos
TOL = 0.004   # graus (~450 m)
try:
    simples = shapely.coverage_simplify(geo.geometry.values, TOL)
except Exception:
    simples = geo.geometry.simplify(TOL, preserve_topology=True).values
geo["geometry"] = simples

x0, y0, x1, y1 = geo.total_bounds
lat_med = np.deg2rad((y0 + y1) / 2)
W = 1000.0
esc = W / ((x1 - x0) * np.cos(lat_med))
H = round((y1 - y0) * esc, 1)

def caminho(geom):
    polys = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
    partes = []
    for p in polys:
        for anel in [p.exterior] + list(p.interiors):
            pts = [f"{(x - x0) * np.cos(lat_med) * esc:.1f},{(y1 - y) * esc:.1f}" for x, y in anel.coords]
            partes.append("M" + "L".join(pts) + "Z")
    return "".join(partes)

geo["svg"] = geo.geometry.map(caminho)
print(f"viewBox 0 0 {W:.0f} {H}  |  {sum(len(s) for s in geo['svg'])/1024:.0f} KB de caminhos")

## 4. Base anonimizada para o site

O nome da empresa serve **só** para identificar empresas distintas. Ele é normalizado, transformado em um
código inteiro sorteado (`SEED_ANONIMO` apenas embaralha; não há função de volta) e descartado. O número do
decreto, a data e o texto livre da planilha também **não** seguem para a página.

In [ ]:
# %% Célula 6 – Base anonimizada (registro por projeto)
nomes_norm = raw["EMPRESA (ESTOQUE)"].map(norm)
unicas = nomes_norm.unique()
perm = np.random.default_rng(SEED_ANONIMO).permutation(len(unicas))
cod = dict(zip(unicas, perm.tolist()))
raw["emp_id"] = nomes_norm.map(cod)

idx_mun = {m: i for i, m in enumerate(geo["mun_norm"])}

def indices(coluna, ordem=None):
    vals = sorted(raw[coluna].unique()) if ordem is None else [o for o in ordem if o in set(raw[coluna])]
    return vals, raw[coluna].map({v: i for i, v in enumerate(vals)}).astype(int)

progs, i_prog = indices("prog", ["PRODEPE", "PROIND"])
anos,  i_ano  = indices("ANO")
segs,  i_seg  = indices("seg")
tipos, i_tipo = indices("tipo", ["Indústria", "Importação", "Central de Distribuição", "PROIND"])
nats,  i_nat  = indices("nat", ["Implantação", "Adesão ou migração", "Ampliação ou revitalização", "Isonomia ou manutenção"])
rds,   i_rd   = indices("rd")
novos, i_novo = indices("novo", ["Novo", "Já existente"])
tipos = [t.replace("Central de Distribuição", "Central de distribuição") for t in tipos]
# "Mais de um segmento", "Outros" e "Não informado" por último
segs_ord = sorted(segs, key=lambda s: (s in ("Mais de um segmento", "Outros", "Não informado (PROIND)"), s))
troca = {segs.index(s): k for k, s in enumerate(segs_ord)}
i_seg = i_seg.map(troca); segs = segs_ord

base = pd.DataFrame({
    "m":    raw["mun_norm"].map(idx_mun).astype(int),
    "prog": i_prog, "ano": i_ano, "seg": i_seg, "tipo": i_tipo, "nat": i_nat, "rd": i_rd, "novo": i_novo,
    "emp":  raw["emp_id"].astype(int),
    "inv":  raw["inv_real_mi"].round(4),
    "job":  raw["job"],
})
COLUNAS_PERMITIDAS = ["m", "prog", "ano", "seg", "tipo", "nat", "rd", "novo", "emp", "inv", "job"]
assert list(base.columns) == COLUNAS_PERMITIDAS

# nomes reais usados só nas conferências – nunca vão para o arquivo
NOMES_PROIBIDOS = set(n for n in nomes_norm.unique() if len(n) >= 6)

print(f"{len(base)} projetos | {base['emp'].nunique()} empresas distintas | {base['m'].nunique()} municípios com projeto")

## 5. Geração do site

O HTML é um único arquivo: CSS, JavaScript, malha, dados e o logo da SDEC (`docs/assets/`, em versão clara e escura) vão embutidos. O mapa é SVG puro (sem
bibliotecas externas); a única chamada externa é a fonte tipográfica, com alternativas locais caso não carregue.
Os filtros recalculam, no navegador, empresas distintas, projetos, investimento e empregos por município
e repintam o mapa e o ranking a cada clique.

In [ ]:
# %% Célula 7 – Modelo do HTML
TEMPLATE = r'''<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>PRODEPE e PROIND em Pernambuco – mapa interativo</title>
<meta name="description" content="Mapa interativo dos projetos aprovados no PRODEPE e no PROIND, por município de Pernambuco: empresas, investimento e empregos.">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
:root{
  --bg:#141414; --tx:#ffffff; --su:#b5b5b5; --su2:#8c8c8c; --ausente:#3a3a3a; --painel:#1b1b1b; --linha:#2c2c2c;
  --cborda:#444; --chover:#888; --on-bg:#e9e9e9; --on-tx:#141414; --hover:#262626; --trilho:#2a2a2a; --sub:#222; --tipbg:#000000dd; --tipborda:#555;
  --fonte:"Inter",system-ui,-apple-system,"Segoe UI",Roboto,"Helvetica Neue",Arial,sans-serif;
  color-scheme:dark;
}
[data-theme="light"]{
  --bg:#f6f5f2; --tx:#141414; --su:#5c5c5c; --su2:#737373; --ausente:#dcdcdc; --painel:#ffffff; --linha:#dedcd6;
  --cborda:#c4c2bc; --chover:#6b6b6b; --on-bg:#141414; --on-tx:#ffffff; --hover:#efeeea; --trilho:#e6e4de; --sub:#faf9f7; --tipbg:#ffffffee; --tipborda:#b5b5b5;
  color-scheme:light;
}
*{box-sizing:border-box}
html,body{margin:0;background:var(--bg);color:var(--tx);font-family:var(--fonte);-webkit-font-smoothing:antialiased}
body{padding:22px 24px 44px}
.wrap{max-width:1280px;margin:0 auto}
button{font-family:inherit}
:focus-visible{outline:2px solid var(--tx);outline-offset:2px}

/* ---------- topo: logo + tema */
.topo{display:flex;justify-content:space-between;align-items:center;gap:16px;margin-bottom:22px}
.logo{height:68px;width:auto;display:block}
.tema{display:inline-flex;border:1px solid var(--cborda);border-radius:999px;overflow:hidden;flex:none}
.tema button{all:unset;cursor:pointer;width:38px;height:32px;display:flex;align-items:center;justify-content:center;color:var(--su)}
.tema button:hover{color:var(--tx)}
.tema button.on{background:var(--on-bg);color:var(--on-tx)}
.tema button[aria-pressed]{}
.tema button:focus-visible{outline:2px solid var(--tx);outline-offset:-2px}

/* ---------- título à esquerda */
header{margin-bottom:20px}
h1{font-weight:700;font-size:clamp(23px,3vw,34px);line-height:1.15;letter-spacing:-.022em;margin:0 0 8px;max-width:900px}
#st{margin:0;color:var(--su);font-size:clamp(14px,1.5vw,16px);line-height:1.45}

/* ---------- grade: conteúdo à esquerda, filtros à direita */
main{display:grid;grid-template-columns:minmax(0,1fr) 308px;gap:22px;align-items:start}
main>*{min-width:0}
aside{background:var(--painel);border:1px solid var(--linha);border-radius:14px;padding:16px;position:sticky;top:16px;max-height:calc(100vh - 32px);overflow-y:auto}
.fecha,#abrir{display:none}
.tit-filtros{font-weight:600;font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:var(--su);margin:0 0 12px}

/* ---------- filtros em lista suspensa */
.campo{margin-bottom:8px}
.cbtn{all:unset;box-sizing:border-box;display:flex;width:100%;align-items:center;gap:10px;cursor:pointer;background:var(--sub);border:1px solid var(--cborda);border-radius:11px;padding:8px 12px;min-height:48px}
.cbtn:hover{border-color:var(--chover)}
.cbtn:focus-visible{outline:2px solid var(--tx);outline-offset:1px}
.campo.ativo .cbtn{border-color:var(--tx)}
.cbtn .txt{flex:1;min-width:0;display:flex;flex-direction:column;gap:2px}
.clab{font-size:10.5px;font-weight:500;letter-spacing:.06em;text-transform:uppercase;color:var(--su2)}
.cval{font-size:14px;font-weight:500;color:var(--tx);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.chev{flex:none;color:var(--su);transition:transform .2s}
.campo.aberto .chev{transform:rotate(180deg)}
.cpainel{margin-top:6px;border:1px solid var(--linha);border-radius:11px;background:var(--painel);padding:6px;max-height:290px;overflow:auto}
.cpainel[hidden]{display:none}
.cacoes{display:flex;gap:16px;padding:6px 8px 8px;border-bottom:1px solid var(--linha);margin-bottom:4px}
.cacoes button{all:unset;cursor:pointer;font-size:12.5px;color:var(--su);text-decoration:underline;text-underline-offset:2px}
.cacoes button:hover{color:var(--tx)}
.dica{margin:0;padding:2px 8px 8px;font-size:11.5px;line-height:1.4;color:var(--su2)}
.opt{display:flex;gap:11px;align-items:center;padding:9px 8px;border-radius:8px;cursor:pointer;font-size:13.5px;line-height:1.3}
.opt:hover{background:var(--hover)}
.opt input{accent-color:#3b82c4;width:17px;height:17px;flex:none;margin:0;cursor:pointer}
.opcoes{margin-top:16px;padding-top:14px;border-top:1px solid var(--linha);display:flex;flex-direction:column;gap:12px;font-size:13px;color:var(--su)}
.opcoes label{display:flex;gap:9px;align-items:center;cursor:pointer}
.opcoes input{accent-color:#3b82c4;width:16px;height:16px;margin:0}
.pal{display:flex;gap:8px;align-items:center}
.pal button{width:44px;height:11px;border:0;border-radius:3px;cursor:pointer;opacity:.55;padding:0}
.pal button.on{opacity:1;outline:1px solid var(--tx);outline-offset:2px}
#reset{all:unset;cursor:pointer;color:var(--su);font-size:13px;text-decoration:underline;text-underline-offset:2px}
#reset:hover{color:var(--tx)}

/* ---------- KPIs e mapa */
#kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:12px}
.kpi{background:var(--painel);border:1px solid var(--linha);border-radius:12px;padding:12px 14px}
.kpi b{display:block;font-weight:700;letter-spacing:-.02em;font-size:clamp(19px,2.2vw,26px);margin-bottom:3px}
.kpi span{font-size:12px;color:var(--su)}
.kpi.ativo{border-color:var(--su)}
#ind-slot{margin-bottom:12px}
#ind-slot:empty{display:none}
#mapwrap{min-width:0;position:relative;background:var(--bg);border:1px solid var(--linha);border-radius:14px;padding:10px 12px 8px}
.mapatopo{display:flex;align-items:center;justify-content:space-between;gap:12px;flex-wrap:wrap;margin-bottom:6px}
.legenda{display:flex;align-items:center;gap:9px;color:var(--su);font-size:12.5px;font-weight:500;flex-wrap:wrap}
.legenda .un{color:var(--su2);font-weight:400;margin-right:4px}
#bar{width:190px;height:9px;border-radius:2px}
.zoom{display:flex;gap:6px}
.zoom button{all:unset;cursor:pointer;width:32px;height:32px;display:flex;align-items:center;justify-content:center;border:1px solid var(--cborda);background:var(--painel);color:var(--tx);border-radius:8px;font-size:18px;line-height:1}
.zoom button:hover{border-color:var(--chover)}
.zoom button:disabled{opacity:.35;cursor:default}
#scroller{overflow-x:auto;overflow-y:hidden;-webkit-overflow-scrolling:touch;border-radius:6px}
#mapa{width:100%;height:auto;display:block}
#mapa path{stroke:var(--bg);stroke-width:.7;cursor:pointer;transition:fill .25s}
#mapa path:hover{stroke:var(--tx);stroke-width:1.3}
#mapa path.sel{stroke:var(--tx);stroke-width:2}
.dicamapa{display:none;margin:6px 4px 2px;font-size:12px;color:var(--su2)}
#tip{position:absolute;pointer-events:none;background:var(--tipbg);border:1px solid var(--tipborda);border-radius:10px;padding:9px 12px;font-size:12.5px;line-height:1.5;min-width:170px;display:none;z-index:5;backdrop-filter:blur(2px)}
#tip strong{font-weight:600;font-size:14px}
#tip .r{color:var(--su)}
#tip .d{display:flex;justify-content:space-between;gap:16px}
#tip .d.a{color:var(--tx);font-weight:600}
#tip .d:not(.a){color:var(--su)}
#baixo{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-top:12px}
.painel{background:var(--painel);border:1px solid var(--linha);border-radius:14px;padding:14px 16px}
.painel h3{font-weight:600;font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:var(--su);margin:0 0 10px}
.rk{display:grid;grid-template-columns:22px minmax(0,1.1fr) minmax(0,1.4fr) auto;gap:8px;align-items:center;padding:3px 4px;border-radius:6px;cursor:pointer;font-size:13px}
.rk:hover,.rk.sel{background:var(--hover)}
.rk i{font-style:normal;color:var(--su);font-size:12px}
.rk .n{white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.rk .b{height:6px;background:var(--trilho);border-radius:3px;overflow:hidden}
.rk .b u{display:block;height:100%;background:var(--cor,#9ecae1);text-decoration:none}
.rk .v{color:var(--su);font-size:12.5px;min-width:64px;text-align:right}
.vazio{color:var(--su);font-size:13px;margin:0}
.s-nome{font-weight:700;font-size:19px;letter-spacing:-.015em;margin:0}
.s-rd{color:var(--su);font-size:12.5px;margin:0 0 10px}
.s-grid{display:grid;grid-template-columns:repeat(2,1fr);gap:8px;margin-bottom:12px}
.s-grid div{background:var(--sub);border:1px solid var(--linha);border-radius:9px;padding:7px 10px}
.s-grid b{display:block;font-size:16px;font-weight:700;letter-spacing:-.01em}
.s-grid span{color:var(--su);font-size:11.5px}
table{border-collapse:collapse;width:100%;font-size:12.5px}
th{color:var(--su);font-weight:500;text-align:right;padding:2px 0 4px}
th:first-child,td:first-child{text-align:left}
td{padding:3px 0;border-top:1px solid var(--linha);text-align:right}
footer{margin-top:24px;color:var(--su);font-size:12.5px;line-height:1.7}
footer p{margin:0 0 4px}

/* ---------- celular e tablet */
@media (max-width:1000px){
  body{padding:14px 14px 88px}
  .topo{margin-bottom:16px}
  .logo{height:54px}
  main{grid-template-columns:minmax(0,1fr)}
  #kpis{grid-template-columns:repeat(2,1fr)}
  #baixo{grid-template-columns:1fr}
  #painel-sel{order:-1}
  .dicamapa{display:block}
  aside{position:fixed;inset:0;z-index:30;max-height:none;border-radius:0;border:0;padding:0 16px 28px;transform:translateY(102%);visibility:hidden;transition:transform .28s ease,visibility 0s .28s}
  aside.aberto{transform:none;visibility:visible;transition:transform .28s ease}
  .tit-filtros{display:none}
  .fecha{display:flex;position:sticky;top:0;z-index:2;background:var(--painel);justify-content:space-between;align-items:center;padding:14px 0 12px;margin-bottom:12px;border-bottom:1px solid var(--linha)}
  .fecha b{font-weight:700;font-size:17px}
  .fecha button{all:unset;cursor:pointer;background:var(--on-bg);color:var(--on-tx);padding:9px 16px;border-radius:999px;font-size:14px;font-weight:500}
  #abrir{display:flex;position:fixed;left:50%;bottom:16px;transform:translateX(-50%);z-index:20;align-items:center;gap:8px;border:0;cursor:pointer;background:var(--on-bg);color:var(--on-tx);font:inherit;font-weight:600;font-size:15px;padding:13px 24px;border-radius:999px;box-shadow:0 6px 22px #0006}
  #abrir i{font-style:normal;background:var(--on-tx);color:var(--on-bg);border-radius:999px;min-width:20px;height:20px;padding:0 6px;font-size:12px;display:none;align-items:center;justify-content:center}
  #abrir i.v{display:inline-flex}
  body.travado{overflow:hidden}
  .cbtn{min-height:52px}
  .opt{padding:11px 8px}
  .cpainel{max-height:min(340px,52vh)}
}
@media (max-width:700px){
  .logo{height:46px}
  #un{display:none}
  .legenda{flex-wrap:nowrap}
  .kpi{padding:10px 12px}
  #mapwrap{padding:8px 8px 6px}
  #bar{width:140px}
  .rk{grid-template-columns:20px minmax(0,1fr) minmax(0,.8fr) auto;font-size:13.5px;padding:6px 4px}
  .painel{padding:12px 13px}
  footer{font-size:12px}
}
</style>
</head>
<body>
<div class="wrap">
<div class="topo">
  <img id="logo" class="logo" alt="Secretaria de Desenvolvimento Econômico – Governo de Pernambuco">
  <div class="tema" id="tema" role="group" aria-label="Tema"><button data-t="light" title="Tema claro" aria-label="Tema claro"><svg width="16" height="16" viewBox="0 0 16 16" fill="none" stroke="currentColor" stroke-width="1.6" stroke-linecap="round" aria-hidden="true"><circle cx="8" cy="8" r="2.9"/><path d="M8 1.5v1.6M8 12.9v1.6M1.5 8h1.6M12.9 8h1.6M3.4 3.4l1.1 1.1M11.5 11.5l1.1 1.1M12.6 3.4l-1.1 1.1M4.5 11.5l-1.1 1.1"/></svg></button><button data-t="dark" title="Tema escuro" aria-label="Tema escuro"><svg width="16" height="16" viewBox="0 0 16 16" fill="none" stroke="currentColor" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M13.2 9.6A5.7 5.7 0 0 1 6.4 2.8a5.7 5.7 0 1 0 6.8 6.8z"/></svg></button></div>
</div>
<header>
  <h1 id="t"></h1>
  <p id="st"></p>
</header>
<main>
  <section>
    <div id="ind-slot"></div>
    <div id="kpis"></div>
    <div id="mapwrap">
      <div class="mapatopo">
        <div class="legenda"><span class="un" id="un"></span><span id="vmin"></span><div id="bar"></div><span id="vmax"></span></div>
        <div class="zoom"><button id="zm" aria-label="Reduzir o mapa">&minus;</button><button id="zp" aria-label="Ampliar o mapa">+</button></div>
      </div>
      <div id="scroller"><svg id="mapa" viewBox="0 0 __W__ __H__" role="img" aria-label="Mapa dos municípios de Pernambuco"></svg></div>
      <div id="tip"></div>
    </div>
    <p class="dicamapa">Toque em um município para ver os valores. Deslize para os lados ou use + e − para ampliar.</p>
    <div id="baixo">
      <div class="painel"><h3 id="rk-t">Maiores municípios</h3><div id="rank"></div></div>
      <div class="painel" id="painel-sel"><h3>Município selecionado</h3><div id="sel"><p class="vazio">Clique em um município no mapa ou no ranking para ver o detalhamento.</p></div></div>
    </div>
  </section>
  <aside id="filtros" aria-label="Filtros">
    <div class="fecha"><b>Filtros</b><button id="fechar">Ver mapa</button></div>
    <p class="tit-filtros">Filtros</p>
    <div id="campos"></div>
    <div class="opcoes">
      <label><input type="checkbox" id="lin"> Escala linear (sem realce de contraste)</label>
      <div class="pal" id="pal"><span>Cores</span></div>
      <button id="reset">Limpar todos os filtros</button>
    </div>
  </aside>
</main>
<footer id="rodape"></footer>
<button id="abrir" aria-controls="filtros">Filtros <i id="nfil"></i></button>
</div>

<script>
const D = __DATA__;
const LOGO = {light:'__LOGO_CLARO__', dark:'__LOGO_ESCURO__'};
const $ = s => document.querySelector(s);
const nf = (x,c=0) => Number(x).toLocaleString('pt-BR',{minimumFractionDigits:c,maximumFractionDigits:c});
const mi = v => v===0 ? '0' : v>=1000 ? nf(v/1000,1)+' bi' : (v>=100 ? nf(v,0)+' mi' : nf(v,1)+' mi');
const IND = {
  emp:  {nome:'Número de empresas', titulo:'Empresas com projetos aprovados', un:'número de empresas', f:v=>nf(v),        cur:v=>nf(v)},
  proj: {nome:'Número de projetos', titulo:'Projetos aprovados',              un:'número de projetos', f:v=>nf(v),        cur:v=>nf(v)},
  inv:  {nome:'Investimento previsto', titulo:'Investimento previsto',        un:'R$ milhões de '+D.anoBase, f:v=>'R$ '+mi(v), cur:mi},
  job:  {nome:'Empregos gerados',   titulo:'Empregos gerados',                un:'número de empregos', f:v=>nf(v),        cur:v=>nf(v)}
};
const ORD = ['emp','proj','inv','job'];
const R = {m:0,prog:1,ano:2,seg:3,tipo:4,nat:5,rd:6,novo:7,emp:8,inv:9,job:10};

// ---------- estado
let indic = 'inv', linear = false, pal = 0, selMun = null;
const sel = {};
D.filtros.forEach(f => sel[f.key] = new Set(f.opts.map((_,i)=>i)));
const html = document.documentElement;
const tema = () => html.dataset.theme;

// ---------- tema
function lerTema(){ try{ const t=localStorage.getItem('tema'); if(t==='light'||t==='dark') return t; }catch(e){} return window.matchMedia && matchMedia('(prefers-color-scheme: light)').matches ? 'light' : 'dark'; }
function setTema(t, salvar){
  html.dataset.theme=t;
  if(salvar){ try{localStorage.setItem('tema',t);}catch(e){} }
  document.querySelectorAll('#tema button').forEach(b=>{ b.classList.toggle('on', b.dataset.t===t); b.setAttribute('aria-pressed', b.dataset.t===t); });
  const lg=$('#logo'); if(LOGO[t]){ lg.src=LOGO[t]; lg.style.display=''; } else lg.style.display='none';
}
setTema(lerTema(), false);
$('#tema').addEventListener('click', e=>{ const t=(e.target.closest('button')||{dataset:{}}).dataset.t; if(t){ setTema(t,true); atualizar(); } });

// ---------- paleta
const hex2rgb = h => [1,3,5].map(i=>parseInt(h.slice(i,i+2),16));
function cor(t){
  const P = D.paletas[pal][tema()==='light' ? 'claro' : 'cores'], x = Math.min(Math.max(t,0),1)*(P.length-1), i = Math.floor(x), j = Math.min(i+1,P.length-1), k = x-i;
  const a = hex2rgb(P[i]), b = hex2rgb(P[j]);
  return 'rgb('+a.map((v,n)=>Math.round(v+(b[n]-v)*k)).join(',')+')';
}
const gama = () => linear ? 1 : D.gama;

// ---------- mapa
$('#mapa').innerHTML = D.paths.map((d,i)=>'<path data-i="'+i+'" d="'+d+'"></path>').join('');
const P = [...document.querySelectorAll('#mapa path')];

// ---------- filtros em lista suspensa
const CHEV = '<svg class="chev" width="16" height="16" viewBox="0 0 16 16" fill="none" stroke="currentColor" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M3.5 6l4.5 4.5L12.5 6"/></svg>';
const CAMPOS = [{key:'ind', label:'Indicador', um:true, opts:ORD.map(k=>IND[k].nome)}].concat(D.filtros.map(f=>Object.assign({um:false},f)));
$('#campos').innerHTML = CAMPOS.map(c=>{
  const acoes = c.um ? '' : '<div class="cacoes"><button data-a="todos">Marcar todos</button><button data-a="nenhum">Limpar</button></div>';
  const dica = c.dica ? '<p class="dica">'+c.dica+'</p>' : '';
  const ops = c.opts.map((o,i)=>'<label class="opt"><input type="'+(c.um?'radio':'checkbox')+'" name="'+c.key+'" data-i="'+i+'"><span>'+o+'</span></label>').join('');
  return '<div class="campo" data-c="'+c.key+'"><button class="cbtn" aria-expanded="false" aria-haspopup="true"><span class="txt"><span class="clab">'+c.label+'</span><span class="cval"></span></span>'+CHEV+'</button><div class="cpainel" hidden>'+acoes+dica+ops+'</div></div>';
}).join('');

const campoEl = k => document.querySelector('.campo[data-c="'+k+'"]');
function fecha(exceto){ document.querySelectorAll('.campo.aberto').forEach(c=>{ if(c!==exceto){ c.classList.remove('aberto'); c.querySelector('.cbtn').setAttribute('aria-expanded','false'); c.querySelector('.cpainel').hidden=true; } }); }
document.addEventListener('click', e=>{
  const btn = e.target.closest('.cbtn');
  if(btn){ const c=btn.parentNode, abre=!c.classList.contains('aberto'); fecha(c); c.classList.toggle('aberto',abre); btn.setAttribute('aria-expanded',abre); c.querySelector('.cpainel').hidden=!abre; return; }
  const ac = e.target.closest('[data-a]');
  if(ac){ const key=ac.closest('.campo').dataset.c, f=D.filtros.find(x=>x.key===key); sel[key]= ac.dataset.a==='todos' ? new Set(f.opts.map((_,i)=>i)) : new Set(); atualizar(); return; }
  if(!e.target.closest('.campo') && !e.target.closest('#abrir')) fecha();
});
document.addEventListener('change', e=>{
  const inp=e.target; if(inp.tagName!=='INPUT' || !inp.closest('.campo')) return;
  const key=inp.closest('.campo').dataset.c, i=+inp.dataset.i;
  if(key==='ind'){ indic=ORD[i]; fecha(); }
  else { inp.checked ? sel[key].add(i) : sel[key].delete(i); }
  atualizar();
});

function resumo(f){
  const n=sel[f.key].size, t=f.opts.length;
  if(n===t) return 'Todos'; if(n===0) return 'Nenhum';
  if(n===1) return f.opts[[...sel[f.key]][0]];
  return n+' de '+t+' selecionados';
}
function pintaCampos(){
  const ci = campoEl('ind'); ci.querySelector('.cval').textContent = IND[indic].nome;
  ci.querySelectorAll('input').forEach((x,i)=>x.checked = ORD[i]===indic);
  D.filtros.forEach(f=>{
    const c=campoEl(f.key); c.querySelector('.cval').textContent = resumo(f);
    c.classList.toggle('ativo', sel[f.key].size!==f.opts.length);
    c.querySelectorAll('input').forEach((x,i)=>x.checked = sel[f.key].has(i));
  });
}
// no celular o seletor de indicador sai da gaveta e fica acima dos totais
const mq = matchMedia('(max-width:1000px)');
function posInd(){ const c=campoEl('ind'); if(mq.matches) $('#ind-slot').appendChild(c); else $('#campos').prepend(c); }
posInd(); mq.addEventListener('change', posInd);

$('#lin').addEventListener('change', e=>{linear=e.target.checked; atualizar();});
$('#pal').insertAdjacentHTML('beforeend', D.paletas.map((p,i)=>'<button data-p="'+i+'" title="'+p.nome+'" aria-label="Cor '+p.nome+'"></button>').join(''));
$('#pal').addEventListener('click', e=>{ if(e.target.dataset.p!==undefined){pal=+e.target.dataset.p; atualizar();} });
$('#reset').addEventListener('click', ()=>{ D.filtros.forEach(f=>sel[f.key]=new Set(f.opts.map((_,i)=>i))); linear=false; $('#lin').checked=false; atualizar(); });

// ---------- agregação
function passa(r){ for (const f of D.filtros) if(!sel[f.key].has(r[R[f.key]])) return false; return true; }
let AGG = [], TOT = null;
function agrega(){
  AGG = new Array(D.mun.length).fill(null);
  const T = {emps:new Set(), proj:0, inv:0, job:0, mun:0};
  for (const r of D.rec){
    if(!passa(r)) continue;
    let a = AGG[r[R.m]]; if(!a) a = AGG[r[R.m]] = {emps:new Set(), proj:0, inv:0, job:0};
    a.emps.add(r[R.emp]); a.proj++; a.inv+=r[R.inv]; a.job+=r[R.job];
    T.emps.add(r[R.emp]); T.proj++; T.inv+=r[R.inv]; T.job+=r[R.job];
  }
  T.mun = AGG.filter(Boolean).length; TOT = T;
}
const val = (a,k) => !a ? null : (k==='emp' ? a.emps.size : a[k]);
const FK = key => D.filtros.find(f=>f.key===key);

// ---------- desenho
function atualizar(){
  agrega();
  const I = IND[indic], g = gama();
  const vals = AGG.map(a=>val(a,indic));
  const ok = vals.filter(v=>v!==null);
  const vmin = ok.length?Math.min(...ok):0, vmax = ok.length?Math.max(...ok):0;
  P.forEach((p,i)=>{
    const v = vals[i];
    p.style.fill = v===null ? 'var(--ausente)' : cor(vmax>vmin ? Math.pow((v-vmin)/(vmax-vmin), g) : .5);
    p.classList.toggle('sel', i===selMun);
  });
  if (selMun!==null) P[selMun].parentNode.appendChild(P[selMun]);

  // título à esquerda, legenda junto ao mapa
  const nProg = sel.prog.size, nomeProg = nProg===2 || nProg===0 ? 'PRODEPE e PROIND' : FK('prog').opts[[...sel.prog][0]];
  const fAno = FK('ano'), anos = [...sel.ano].map(i=>fAno.opts[i]).sort();
  $('#t').textContent = I.titulo+' – '+nomeProg+', Pernambuco';
  const partes = ['por município'];
  if (anos.length===fAno.opts.length) partes.push('decretos de '+anos[0]+' a '+anos[anos.length-1]);
  else if (anos.length) partes.push('decretos de '+anos.join(', '));
  partes.push(I.un);
  const ativos = D.filtros.filter(f=>!['prog','ano'].includes(f.key) && sel[f.key].size!==f.opts.length).length;
  if (ativos) partes.push(ativos+(ativos>1?' filtros adicionais ativos':' filtro adicional ativo'));
  $('#st').textContent = partes.join(' · ');
  $('#un').textContent = I.un;
  $('#vmin').textContent = ok.length ? I.cur(vmin) : '–';
  $('#vmax').textContent = ok.length ? I.cur(vmax) : '–';
  const stops = []; for (let k=0;k<=20;k++) stops.push(cor(Math.pow(k/20,g))+' '+(k*5)+'%');
  $('#bar').style.background = 'linear-gradient(90deg,'+stops.join(',')+')';

  // controles
  pintaCampos();
  document.querySelectorAll('#pal button').forEach((b,i)=>{ b.classList.toggle('on', i===pal); b.style.background='linear-gradient(90deg,'+D.paletas[i][tema()==='light'?'claro':'cores'].join(',')+')'; });
  const nAtivos = D.filtros.filter(f=>sel[f.key].size!==f.opts.length).length + (linear?1:0);
  const bi = $('#nfil'); bi.textContent = nAtivos; bi.classList.toggle('v', nAtivos>0);

  // KPIs
  const K = [['emp',TOT.emps.size,'empresas'],['proj',TOT.proj,'projetos'],['inv',TOT.inv,'investimento previsto'],['job',TOT.job,'empregos gerados']];
  $('#kpis').innerHTML = K.map(([k,v,r])=>'<div class="kpi'+(k===indic?' ativo':'')+'"><b>'+IND[k].f(v)+'</b><span>'+r+'</span></div>').join('');

  // ranking
  $('#rk-t').textContent = 'Maiores municípios – '+I.nome.toLowerCase();
  const rk = vals.map((v,i)=>[i,v]).filter(x=>x[1]!==null && x[1]>0).sort((a,b)=>b[1]-a[1]).slice(0,10);
  const top = rk.length?rk[0][1]:1;
  $('#rank').innerHTML = rk.length ? rk.map(([i,v],n)=>'<div class="rk'+(i===selMun?' sel':'')+'" data-i="'+i+'"><i>'+(n+1)+'</i><span class="n">'+D.mun[i]+'</span><span class="b"><u style="width:'+(v/top*100)+'%;--cor:'+cor(Math.pow(vmax>vmin?(v-vmin)/(vmax-vmin):.5,g))+'"></u></span><span class="v">'+I.cur(v)+'</span></div>').join('') : '<p class="vazio">Nenhum município com projetos nesta combinação de filtros.</p>';

  detalhe();
  rodape();
}
function rodape(){
  $('#rodape').innerHTML =
   '<p>Fonte: ADEPE (planilha de novos investimentos, PRODEPE e PROIND) | Elaboração: SDEC-PE / NAIEE.</p>'+
   '<p>Data-base: decretos até '+D.dataBase+'. Valores monetários deflacionados pelo IPCA (IBGE) para preços de '+D.anoBase+', segundo o ano do decreto; em '+D.anoBase+' os dados cobrem apenas parte do ano.</p>'+
   '<p>Em cinza, municípios sem projetos na seleção atual. Empresas contadas uma única vez em cada município (ou no estado, nos totais), mesmo com mais de um projeto. Os dados não identificam as empresas.</p>';
}
function detalhe(){
  const box = $('#sel');
  if (selMun===null){ box.innerHTML='<p class="vazio">Clique em um município no mapa ou no ranking para ver o detalhamento.</p>'; return; }
  const a = AGG[selMun];
  const rdM = D.rec.find(r=>r[R.m]===selMun);
  const rd = rdM ? FK('rd').opts[rdM[R.rd]] : '';
  let h = '<p class="s-nome">'+D.mun[selMun]+'</p><p class="s-rd">'+(rd?'RD '+rd:'')+'</p>';
  if(!a){ box.innerHTML = h+'<p class="vazio">Sem projetos nesta combinação de filtros.</p>'; return; }
  h += '<div class="s-grid"><div><b>'+nf(a.emps.size)+'</b><span>empresas</span></div><div><b>'+nf(a.proj)+'</b><span>projetos</span></div><div><b>R$ '+mi(a.inv)+'</b><span>investimento previsto</span></div><div><b>'+nf(a.job)+'</b><span>empregos gerados</span></div></div>';
  const porAno = {}, porSeg = {};
  for (const r of D.rec){ if(r[R.m]!==selMun||!passa(r)) continue;
    const y=FK('ano').opts[r[R.ano]]; (porAno[y]=porAno[y]||{p:0,i:0,j:0}); porAno[y].p++; porAno[y].i+=r[R.inv]; porAno[y].j+=r[R.job];
    const s=FK('seg').opts[r[R.seg]]; (porSeg[s]=porSeg[s]||{p:0,i:0}); porSeg[s].p++; porSeg[s].i+=r[R.inv]; }
  h += '<table><tr><th>Ano</th><th>Projetos</th><th>Invest.</th><th>Empregos</th></tr>'+Object.keys(porAno).sort().map(y=>'<tr><td>'+y+'</td><td>'+porAno[y].p+'</td><td>'+mi(porAno[y].i)+'</td><td>'+nf(porAno[y].j)+'</td></tr>').join('')+'</table>';
  h += '<table style="margin-top:10px"><tr><th>Segmento</th><th>Projetos</th><th>Invest.</th></tr>'+Object.entries(porSeg).sort((x,y)=>y[1].p-x[1].p).slice(0,5).map(([s,v])=>'<tr><td>'+s+'</td><td>'+v.p+'</td><td>'+mi(v.i)+'</td></tr>').join('')+'</table>';
  box.innerHTML = h;
}

// ---------- interação com o mapa
const tip = $('#tip'), wrap = $('#mapwrap');
function dica(i, ev){
  const a = AGG[i];
  let h = '<strong>'+D.mun[i]+'</strong>';
  if(!a) h += '<div class="r">Sem projetos na seleção</div>';
  else h += ORD.map(k=>'<div class="d'+(k===indic?' a':'')+'"><span>'+IND[k].nome+'</span><span>'+IND[k].f(val(a,k))+'</span></div>').join('');
  tip.innerHTML = h; tip.style.display='block';
  const b = wrap.getBoundingClientRect();
  let x = ev.clientX-b.left+14, y = ev.clientY-b.top+14;
  if (x+200>b.width) x = ev.clientX-b.left-200;
  tip.style.left = Math.max(4,x)+'px'; tip.style.top = y+'px';
}
$('#mapa').addEventListener('mousemove', e=>{ const p=e.target.closest('path'); if(p) dica(+p.dataset.i,e); else tip.style.display='none'; });
$('#mapa').addEventListener('mouseleave', ()=>tip.style.display='none');
$('#mapa').addEventListener('click', e=>{ const p=e.target.closest('path'); if(p){ const i=+p.dataset.i; selMun = selMun===i?null:i; atualizar(); } });
$('#rank').addEventListener('click', e=>{ const r=e.target.closest('.rk'); if(r){ selMun=+r.dataset.i; atualizar(); } });
document.addEventListener('click', e=>{ if(!e.target.closest('#mapa')) tip.style.display='none'; });

// ---------- gaveta de filtros (celular)
const gav = $('#filtros');
function gaveta(a){ gav.classList.toggle('aberto',a); document.body.classList.toggle('travado',a); if(a) gav.scrollTop=0; }
$('#abrir').addEventListener('click', ()=>gaveta(true));
$('#fechar').addEventListener('click', ()=>gaveta(false));
document.addEventListener('keydown', e=>{ if(e.key==='Escape'){ gaveta(false); fecha(); } });
matchMedia('(min-width:1001px)').addEventListener('change', e=>{ if(e.matches) gaveta(false); });

// ---------- zoom do mapa
const NIVEIS = [1,2,3,4.5]; let zi = 0;
function zoom(d){
  const sc = $('#scroller'), ant = (sc.scrollLeft + sc.clientWidth/2) / sc.scrollWidth;
  zi = Math.min(NIVEIS.length-1, Math.max(0, zi+d));
  $('#mapa').style.width = (NIVEIS[zi]*100)+'%';
  sc.scrollLeft = ant*sc.scrollWidth - sc.clientWidth/2;
  $('#zm').disabled = zi===0; $('#zp').disabled = zi===NIVEIS.length-1;
}
$('#zp').addEventListener('click', ()=>zoom(1)); $('#zm').addEventListener('click', ()=>zoom(-1)); zoom(0);

atualizar();
</script>
</body>
</html>
'''

In [ ]:
# %% Célula 8 – Gera docs/index.html
FILTROS = [
    {"key": "prog", "label": "Programa",                "idx": 1, "opts": progs},
    {"key": "ano",  "label": "Ano do decreto",          "idx": 2, "opts": [str(a) for a in anos]},
    {"key": "seg",  "label": "Segmento",                "idx": 3, "opts": segs,
     "dica": "O PROIND não informa segmento: para incluí-lo, mantenha marcada a opção Não informado (PROIND)."},
    {"key": "tipo", "label": "Tipo de projeto",         "idx": 4, "opts": tipos},
    {"key": "nat",  "label": "Natureza do projeto",     "idx": 5, "opts": nats},
    {"key": "rd",   "label": "Região de desenvolvimento","idx": 6, "opts": [titulo(r) for r in rds]},
    {"key": "novo", "label": "Empreendimento",          "idx": 7, "opts": novos},
]
DADOS = {
    "mun":   geo["NM_MUN"].tolist(),
    "paths": geo["svg"].tolist(),
    "rec":   base[COLUNAS_PERMITIDAS].values.tolist(),
    "filtros": FILTROS,
    "paletas": PALETAS,
    "gama": GAMA,
    "anoBase": ANO_BASE,
    "dataBase": DATA_BASE,
}
# inteiros como inteiros (o .values.tolist() promove tudo a float por causa de 'inv')
for r in DADOS["rec"]:
    for k in (0, 1, 2, 3, 4, 5, 6, 7, 8, 10):
        r[k] = int(r[k])

import base64
def logo_b64(nome):
    """Logo embutido no HTML (arquivo único). Se faltar, a página simplesmente omite a imagem."""
    p = os.path.join(LOGO_DIR, nome)
    if not os.path.exists(p):
        print(f"AVISO: {p} não encontrado – página sem logo")
        return ""
    return "data:image/png;base64," + base64.b64encode(open(p, "rb").read()).decode()

html = (TEMPLATE.replace("__LOGO_CLARO__", logo_b64("logo_sdec_claro.png"))
                .replace("__LOGO_ESCURO__", logo_b64("logo_sdec_escuro.png"))
                .replace("__DATA__", json.dumps(DADOS, ensure_ascii=False, separators=(",", ":")))
                .replace("__W__", f"{W:.0f}").replace("__H__", f"{H}"))
arq_html = os.path.join(OUT, "index.html")
with open(arq_html, "w", encoding="utf-8") as f:
    f.write(html)
open(os.path.join(OUT, ".nojekyll"), "w").close()
print(f"{arq_html}  ({os.path.getsize(arq_html)/1024:.0f} KB)")

## 6. Verificação antes de publicar

Três conferências que precisam passar: (i) nenhum nome de empresa (nem o nome completo normalizado) aparece
no HTML; (ii) os totais do site batem com os da planilha; (iii) o investimento deflacionado bate com o cálculo
direto ano a ano.

In [ ]:
# %% Célula 9 – Verificação
txt = norm(open(arq_html, encoding="utf-8").read())
vazou = [n for n in NOMES_PROIBIDOS if f" {n} " in f" {txt} "]
assert not vazou, f"{len(vazou)} nome(s) de empresa encontrados no HTML"
print(f"OK: nenhum dos {len(NOMES_PROIBIDOS)} nomes de empresa aparece no HTML")

# o dicionário embutido só tem as colunas previstas
assert set(DADOS) == {"mun", "paths", "rec", "filtros", "paletas", "gama", "anoBase", "dataBase"}
assert all(len(r) == len(COLUNAS_PERMITIDAS) for r in DADOS["rec"])

# totais
tot_planilha = raw.groupby("ANO")["INVESTIMENTOS (R$)"].sum().mul(raw.groupby("ANO")["ANO"].first().map(DEFL)).sum() / 1e6
tot_site     = sum(r[9] for r in DADOS["rec"])
assert abs(tot_planilha - tot_site) < 0.01 * len(DADOS["rec"]) / 1000 + 0.05, (tot_planilha, tot_site)
print(f"OK: investimento total no site = R$ {br(tot_site, 1)} mi de {ANO_BASE}  (planilha: {br(tot_planilha, 1)})")
print(f"OK: {sum(r[10] for r in DADOS['rec'])} empregos | {base['emp'].nunique()} empresas | {len(base)} projetos")